# Milestone 2: Data Cleaning

Melanjutkan `01_data_profiling.ipynb` untuk dataset **Indonesian DFK Domain CPT Corpus** (`by_source_domain`).

Dataset ini berupa korpus teks. Kolom utama yang dibersihkan adalah `text`, sedangkan `subset` menunjukkan asal data (misalnya `news_dfk_formal`, `wikipedia_dfk`, dan `jurnal_dfk`). Karena dataset tidak memiliki kolom numerik pengukuran, penanganan outlier numerik dan clipping tidak diterapkan. Panjang teks digunakan untuk analisis anomali, bukan langsung dihapus.

Alur: memuat Parquet → profiling → missing value → normalisasi teks → pemeriksaan panjang teks → deduplikasi → validasi → menyimpan hasil.


In [ ]:
from pathlib import Path
import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dataset.parquet"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}. "
        "Jalankan 01_data_profiling.ipynb terlebih dahulu."
    )

df = pl.read_parquet(DATA_PATH)
print(f"Ukuran awal: {df.height:,} baris x {df.width} kolom")
print("Kolom:", df.columns)
df.head()


## 1. Profiling awal dengan DuckDB `SUMMARIZE`

Ringkasan tipe data, nilai null, rentang, dan jumlah nilai berbeda untuk memeriksa kondisi dataset sebelum cleaning.


In [ ]:
# Membuat koneksi DuckDB
connection = duckdb.connect()

# Mendaftarkan dataframe ke DuckDB
connection.register("dataset_awal", df)

# Melakukan profiling awal menggunakan DuckDB SUMMARIZE
profil_awal = connection.execute(
    "SUMMARIZE SELECT * FROM dataset_awal"
).fetchdf()

# Menampilkan hasil profiling
profil_awal


## 2. Pemeriksaan missing values

Fokus pemeriksaan adalah kolom `text`. Baris tanpa teks tidak dapat digunakan sebagai dokumen korpus, sehingga akan dihapus. Kolom `subset` juga diperiksa karena diperlukan untuk mengetahui asal dokumen.


In [ ]:
jumlah_baris_awal = df.height

null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name="kolom", column_names=["jumlah_null"])
    .with_columns(
        (pl.col("jumlah_null") / max(jumlah_baris_awal, 1) * 100)
        .round(2).alias("persen_null")
    )
    .sort("jumlah_null", descending=True)
)
null_summary


## 3. Penanganan missing values dan teks kosong

Tidak menggunakan median atau modus karena kolom utama berisi teks. Dokumen dengan `text` null atau hanya spasi dihapus. Baris dengan `subset` kosong juga dihapus agar asal data tetap dapat ditelusuri.


In [ ]:
if "text" not in df.columns:
    raise ValueError("Kolom 'text' tidak ditemukan. Periksa hasil profiling dataset.")

df_clean = df.with_columns(
    pl.col("text").cast(pl.String, strict=False)
)

# Hitung teks null/kosong sebelum penghapusan
teks_null = df_clean.select(pl.col("text").is_null().sum()).item()
teks_kosong = df_clean.filter(
    pl.col("text").is_not_null() &
    (pl.col("text").str.strip_chars().str.len_chars() == 0)
).height

df_clean = df_clean.filter(
    pl.col("text").is_not_null() &
    (pl.col("text").str.strip_chars().str.len_chars() > 0)
)

if "subset" in df_clean.columns:
    df_clean = df_clean.filter(
        pl.col("subset").is_not_null() &
        (pl.col("subset").cast(pl.String).str.strip_chars().str.len_chars() > 0)
    )

print("Text null:", f"{teks_null:,}")
print("Text kosong:", f"{teks_kosong:,}")
print("Baris setelah penanganan missing/empty:", f"{df_clean.height:,}")


## 4. Normalisasi teks

Normalisasi dilakukan secara konservatif: menghapus spasi di awal/akhir dan merapikan rangkaian whitespace menjadi satu spasi. Huruf besar-kecil, tanda baca, angka, nama, dan isi kalimat tidak diubah agar makna dokumen tetap terjaga.


In [ ]:
df_clean = df_clean.with_columns(
    pl.col("text")
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .alias("text")
)

# Tambahkan panjang karakter untuk analisis kualitas teks
df_clean = df_clean.with_columns(
    pl.col("text").str.len_chars().alias("panjang_teks")
)

print("Contoh hasil normalisasi:")
df_clean.select("text", "panjang_teks").head(5)


## 5. Pemeriksaan anomali panjang teks (outlier)

Karena data berupa dokumen teks, panjang teks bukan atribut numerik yang boleh di-*clip*. IQR digunakan untuk menandai dokumen yang sangat pendek/panjang sebagai bahan pemeriksaan. Baris tidak otomatis dihapus hanya karena berada di luar batas IQR.


In [ ]:
q1 = df_clean.select(pl.col("panjang_teks").quantile(0.25)).item()
q3 = df_clean.select(pl.col("panjang_teks").quantile(0.75)).item()
iqr = q3 - q1
batas_bawah = max(0, q1 - 1.5 * iqr)
batas_atas = q3 + 1.5 * iqr

anomali_panjang = df_clean.filter(
    (pl.col("panjang_teks") < batas_bawah) |
    (pl.col("panjang_teks") > batas_atas)
)

print(f"Q1: {q1}")
print(f"Q3: {q3}")
print(f"IQR: {iqr}")
print(f"Batas bawah: {batas_bawah}")
print(f"Batas atas: {batas_atas}")
print(f"Dokumen di luar batas IQR: {anomali_panjang.height:,}")

# Lihat beberapa contoh untuk pemeriksaan manual
anomali_panjang.select("subset", "panjang_teks", "text").head(10)


## 6. Pemeriksaan dan penghapusan duplikasi

Deduplikasi menggunakan isi `text` yang sudah dinormalisasi. Dokumen dengan teks sama dianggap duplikat, sedangkan `subset` dipertahankan sebagai informasi asal pada baris yang pertama ditemukan.


In [ ]:
baris_sebelum_deduplikasi = df_clean.height
jumlah_duplikat = df_clean.height - df_clean.select(
    pl.col("text").n_unique()
).item()

df_dedup = df_clean.unique(
    subset=["text"],
    keep="first",
    maintain_order=True
)

print(f"Baris sebelum deduplikasi: {baris_sebelum_deduplikasi:,}")
print(f"Duplikat berdasarkan text: {jumlah_duplikat:,}")
print(f"Baris setelah deduplikasi: {df_dedup.height:,}")


## 7. Validasi kualitas data menggunakan DuckDB

Validasi memastikan kolom teks tidak null/kosong, panjang teks positif, serta tidak ada teks duplikat. Jika kolom `subset` tersedia, jumlah dokumen per subset juga ditampilkan.


In [ ]:
connection.register("dataset_bersih", df_dedup)

validasi = connection.execute("""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN text IS NULL THEN 1 ELSE 0 END) AS text_null,
        SUM(CASE WHEN TRIM(text) = '' THEN 1 ELSE 0 END) AS text_kosong,
        SUM(CASE WHEN LENGTH(text) = 0 THEN 1 ELSE 0 END) AS panjang_nol,
        COUNT(DISTINCT text) AS text_unik
    FROM dataset_bersih
""").pl()

validasi


In [ ]:
if "subset" in df_dedup.columns:
    ringkasan_subset = (
        df_dedup.group_by("subset")
        .agg(pl.len().alias("jumlah_dokumen"))
        .sort("jumlah_dokumen", descending=True)
    )
    display(ringkasan_subset)
else:
    print("Kolom subset tidak tersedia pada dataset.")


## 8. Perbandingan sebelum dan sesudah cleaning

Ringkasan ini menunjukkan perubahan jumlah baris pada setiap tahapan. Penghapusan dilakukan untuk teks null/kosong dan duplikasi teks; anomali panjang hanya ditandai untuk pemeriksaan.


In [ ]:
ringkasan_cleaning = pl.DataFrame({
    "tahapan": [
        "Data awal",
        "Setelah missing value dan teks kosong",
        "Setelah deduplikasi"
    ],
    "jumlah_baris": [
        jumlah_baris_awal,
        # df_clean sebelum deduplikasi masih tersedia? gunakan jumlah sebelum dedup
        baris_sebelum_deduplikasi,
        df_dedup.height
    ]
})
ringkasan_cleaning


## 9. Simpan dataset bersih

Hasil disimpan ke `data/processed/dataset_bersih.parquet` agar dapat digunakan pada milestone berikutnya.


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH, compression="snappy")

print(f"Baris awal: {jumlah_baris_awal:,}")
print(f"Baris akhir: {df_dedup.height:,}")
print(f"Baris yang dihapus: {jumlah_baris_awal - df_dedup.height:,}")
print(f"Dataset bersih tersimpan di: {OUTPUT_PATH}")


## Catatan Milestone 2

Setelah notebook dijalankan, dokumentasikan:
- Jumlah data awal dan akhir.
- Jumlah teks null/kosong yang dihapus.
- Jumlah duplikasi teks yang dihapus.
- Batas IQR panjang teks dan jumlah dokumen yang ditandai sebagai anomali.
- Distribusi dokumen pada setiap `subset`.
- Alasan mempertahankan dokumen outlier panjang: panjang teks yang ekstrem belum tentu salah atau tidak relevan untuk korpus Continued Pre-Training.

**Catatan:** Dataset ini sudah memiliki tahap preprocessing dan deduplikasi menurut deskripsi dataset. Cleaning pada milestone ini merupakan pemeriksaan ulang pada salinan data yang digunakan dalam pipeline proyek, bukan klaim bahwa dataset mentah belum pernah dibersihkan.
